# Classification and Localization in Class-Imbalanced, Extremely Shifted Images
**IE 643 Course Project — Team Latent Navigators (Ananna Biswas, 25M1727)**

This notebook is the full pipeline after the prep-presentation review. It covers:

| Section | What it does |
|---|---|
| 0–1 | Setup, configuration, paths |
| 2 | PASCAL VOC 2012: download and annotation parsing |
| 3 | Controlled class imbalance: new majority/minority split, minority-first sampler, count audit |
| 4 | Fixed tuning / test split of VOC val |
| 5 | **Real** extreme-condition test sets: RTTS (haze), ExDark (low light), DAWN (fog/rain/snow/sand) |
| 6 | Randomized, spatially varying synthetic degradations (training augmentation + controlled severity tests) |
| 7 | Datasets, loaders and cached evaluation banks |
| 8 | ResNet-50 architecture overview |
| 9–10 | Class-balanced loss and the training loop |
| 11 | Metrics (per-class AP / mAP, tuned-threshold F1) and test-time BatchNorm adaptation |
| 12–14 | Classification ablations: training, evaluation, tables and plots |
| 15 | Qualitative results and Grad-CAM |
| 16–18 | Localization: YOLOv8 on the same imbalanced subset, evaluated on synthetic and real conditions |
| 19 | Novelty preview (off by default) |
| 20 | Interactive demo (Gradio) |
| 21 | Summary export for the report and slides |

**Changes made in response to the prep-review feedback**
* *Natural extreme-condition images*: models are now tested on three real datasets (RTTS, ExDark, DAWN). Synthetic degradations are used only for training augmentation and for a controlled severity sweep.
* *Deterministic atmospheric-scattering haze*: the old haze used a single scalar transmission `t = exp(-beta)` (a uniform grey blend). It is replaced by randomized, spatially varying degradations (pseudo-depth + fractal heterogeneity, random airlight colour, sensor noise), and the synthetic-vs-real gap is measured explicitly.
* *ResNet-50 overview*: Section 8 prints the stage-by-stage architecture and draws a diagram for the slides.



## 0. Setup

In [ ]:
import os, sys, subprocess, importlib.util

IN_COLAB = 'google.colab' in sys.modules
IN_NOTEBOOK = 'ipykernel' in sys.modules


def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)


_required = {'ultralytics': 'ultralytics', 'gdown': 'gdown', 'tqdm': 'tqdm', 'sklearn': 'scikit-learn', 'pandas': 'pandas'}
_missing = [pkg for mod, pkg in _required.items() if importlib.util.find_spec(mod) is None]
if _missing:
    pip_install(*_missing)

import json, math, re, time, copy, random, shutil, zipfile, tarfile, urllib.request
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageFile
import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from sklearn.metrics import average_precision_score

cv2.setNumThreads(0)  # avoid oversubscription inside DataLoader workers
ImageFile.LOAD_TRUNCATED_IMAGES = True  # a few files in the real datasets are truncated
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| numpy', np.__version__)

## 1. Configuration and paths

**Real-world datasets.** ExDark downloads automatically (official Google Drive links from the dataset's GitHub page). RTTS and DAWN need one manual download each; put the zip (or the extracted folder) anywhere and set the path below. Any dataset that is missing is skipped with a warning, so the rest of the pipeline still runs.
* **RTTS** (RESIDE-β, real hazy traffic images, VOC-format XML): from the RESIDE project page, or any mirror that keeps the VOC XML annotations.
* **DAWN** (real fog / rain / snow / sandstorm): Mendeley Data, DOI 10.17632/766ygrbt8y ("Download all").

In [ ]:
QUICK_RUN = False   # True = tiny end-to-end dry run to check the whole pipeline before the full run
USE_DRIVE = True    # Colab: keep checkpoints/results on Google Drive so a disconnect loses nothing

CFG = dict(
    # data and imbalance
    data_seed=0,
    seeds=[0, 1, 2],                                       # training seeds per configuration
    majority=['person', 'car', 'chair', 'bottle', 'dog'],
    minority=['bicycle', 'motorbike', 'bus', 'boat', 'cat'],
    majority_n=300, minority_n=30,                         # target image-level positives per class (10:1)
    n_tune=1000,                                           # VOC val images held out for tuning; the rest is test
    # classification training
    pretrained=True, img_size=224, max_side=640,
    batch_size=32, epochs=12, lr=1e-4, weight_decay=1e-4, warmup_epochs=1,
    num_workers=2,
    aug_prob=0.6,                                          # P(training image is degraded) in augmentation configs
    weighting='cb_pos', cb_beta=0.999, max_pos_weight=20.0,
    consistency_weight=1.0,                                # novelty preview only (Section 19)
    # synthetic degradations
    degr_types=['haze', 'lowlight', 'snow', 'rain', 'sandstorm'],
    severities=[1, 2, 3],
    syn_eval_n=1500,                                       # VOC test images used for the synthetic severity sweep
    # real-world test sets
    exdark_src=None,                                       # None = auto-download; or a path to zip/folder
    rtts_src='/content/drive/MyDrive/datasets/RTTS.zip',
    dawn_src='/content/drive/MyDrive/datasets/DAWN.zip',
    urhi_src=None,                                         # optional: RESIDE unannotated real hazy images (BN adaptation only)
    real_adapt_frac=0.15, real_adapt_n=500,                # unlabeled images per real dataset for BN adaptation
    max_real_test=None,
    bn_adapt_batch=64,
    # detection
    yolo_model='yolov8n.pt', yolo_epochs=40, yolo_imgsz=640, yolo_batch=16,
    rfs_t=0.2,                                             # repeat-factor sampling threshold
    det_configs=['baseline', 'combined'],                  # add 'oversample', 'augment' for the full 4-row ablation
    det_syn_n=500, det_severities=[1, 2, 3],
    # demo
    launch_demo=True,
)
if QUICK_RUN:
    CFG.update(seeds=[0], epochs=2, majority_n=60, minority_n=8, n_tune=200, syn_eval_n=150,
               severities=[3], real_adapt_n=64, max_real_test=200, yolo_epochs=2, yolo_imgsz=320,
               det_syn_n=50, det_severities=[3])
CFG.update(json.loads(os.environ.get('IE643_OVERRIDES', '{}')))  # optional overrides without editing this cell

MAJORITY, MINORITY = CFG['majority'], CFG['minority']
CLASSES = MAJORITY + MINORITY
CIDX = {c: i for i, c in enumerate(CLASSES)}
NC = len(CLASSES)
MAJ_IDX = [CIDX[c] for c in MAJORITY]
MIN_IDX = [CIDX[c] for c in MINORITY]

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK = Path('/content/drive/MyDrive/IE643_LatentNavigators')
    else:
        WORK = Path('/content/ie643_work')
    DATA = Path('/content/data')
else:
    WORK = Path(os.environ.get('IE643_WORK', 'ie643_work')).resolve()
    DATA = Path(os.environ.get('IE643_DATA', 'ie643_data')).resolve()
if QUICK_RUN:
    WORK = WORK / 'quick_run'
CACHE = DATA / ('cache_quick' if QUICK_RUN else 'cache')
YOLO_DIR = DATA / ('yolo_quick' if QUICK_RUN else 'yolo')
CKPT, RES, FIG, PRED = WORK / 'checkpoints', WORK / 'results', WORK / 'figures', WORK / 'predictions'
YOLO_RUNS, DEMO_DIR = WORK / 'yolo_runs', WORK / 'demo_examples'
for p in [DATA, CACHE, YOLO_DIR, CKPT, RES, FIG, PRED, YOLO_RUNS, DEMO_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = device.type == 'cuda'
torch.backends.cudnn.benchmark = True
HW = dict(device=str(device), gpu=torch.cuda.get_device_name(0) if AMP else None,
          cpu_count=os.cpu_count(), torch=torch.__version__, torchvision=torchvision.__version__)
print(HW)
print('Work dir:', WORK, '| Data dir:', DATA)
(RES / 'run_config.json').write_text(json.dumps(dict(CFG=CFG, hardware=HW), indent=2))

# Chart styling (light surface; fixed categorical order so a config keeps its colour in every figure)
INK, INK2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'axes.edgecolor': GRID,
                     'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2, 'text.color': INK,
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10, 'legend.frameon': False})


def savefig(fig, name):
    fig.savefig(FIG / name, dpi=200, bbox_inches='tight')
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)

## 2. PASCAL VOC 2012
Annotations are parsed straight from the XML files (no image decoding), which is much faster than iterating `VOCDetection`. Each image becomes a *record*: path, image-level labels, and boxes `(class, x1, y1, x2, y2, difficult)`. Class names from every dataset go through one synonym map (e.g. ExDark `People` → `person`, DAWN `motorcycle` → `motorbike`).

In [ ]:
VOC20 = ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable',
         'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']
NAME_MAP = {c: c for c in VOC20}
NAME_MAP.update({'people': 'person', 'pedestrian': 'person', 'persons': 'person', 'human': 'person',
                 'table': 'diningtable', 'dining table': 'diningtable', 'motorcycle': 'motorbike',
                 'motor': 'motorbike', 'airplane': 'aeroplane', 'plane': 'aeroplane', 'tv': 'tvmonitor'})
UNMAPPED = defaultdict(Counter)   # names we could not map, per source (e.g. ExDark 'cup', DAWN 'truck')


def map_name(raw, source):
    key = raw.strip().lower()
    c = NAME_MAP.get(key)
    if c is None:
        UNMAPPED[source][key] += 1
    return c


def make_record(path, objs, source, meta=None, w=None, h=None):
    mapped = []
    for o in objs:
        c = map_name(o['name'], source)
        if c is not None:
            mapped.append([c, *[float(v) for v in o['box']], int(o.get('difficult', 0))])
    return dict(path=str(path), objects=mapped, labels=sorted({m[0] for m in mapped}),
                w=w, h=h, source=source, meta=meta or {})


def parse_voc_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    fn = (root.findtext('filename') or '').strip()
    size = root.find('size')
    w = h = None
    if size is not None:
        try:
            w, h = int(float(size.findtext('width'))), int(float(size.findtext('height')))
        except (TypeError, ValueError):
            w = h = None
    objs = []
    for o in root.iter('object'):
        bb = o.find('bndbox')
        if bb is None:
            continue
        try:
            box = [float(bb.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax')]
        except (TypeError, ValueError):
            continue
        objs.append(dict(name=o.findtext('name') or '', box=box,
                         difficult=int(float(o.findtext('difficult') or 0))))
    return fn, (w if w else None), (h if h else None), objs


VOC_ROOT = DATA / 'VOCdevkit' / 'VOC2012'
VOC_MIRROR = 'https://pjreddie.com/media/files/VOCtrainval_11-May-2012.tar'


def download_voc():
    if (VOC_ROOT / 'Annotations').exists():
        return
    try:
        torchvision.datasets.VOCDetection(root=str(DATA), year='2012', image_set='trainval', download=True)
    except Exception as e:
        print('Official VOC server failed (%s); trying mirror' % e)
        tar_path = DATA / 'VOCtrainval_11-May-2012.tar'
        urllib.request.urlretrieve(VOC_MIRROR, tar_path)
        with tarfile.open(tar_path) as tf:
            tf.extractall(DATA)


def load_voc_split(split):
    cache = CACHE / f'voc_{split}_records.json'
    if cache.exists():
        return json.loads(cache.read_text())
    ids = (VOC_ROOT / 'ImageSets' / 'Main' / f'{split}.txt').read_text().split()
    recs = []
    for i in tqdm(ids, desc=f'parse VOC {split}'):
        _, w, h, objs = parse_voc_xml(VOC_ROOT / 'Annotations' / f'{i}.xml')
        recs.append(make_record(VOC_ROOT / 'JPEGImages' / f'{i}.jpg', objs, 'voc', {'id': i}, w, h))
    cache.write_text(json.dumps(recs))
    return recs


download_voc()
voc_train_all = load_voc_split('train')
voc_val_all = load_voc_split('val')
print(f'VOC 2012 train: {len(voc_train_all)} images | val: {len(voc_val_all)} images')

## 3. Controlled class imbalance
**Class split.** The 10 classes are chosen so that every one of them also appears in a real extreme-condition test set: all 10 are in ExDark, and RTTS covers 2 majority (person, car) and 3 minority classes (bicycle, motorbike, bus). The classifier has a 10-way head. In the earlier run, untargeted classes such as *bird* only entered training through co-occurrence, which is why their F1 was near zero.

**Sampler.** The earlier sampler asked for 300 vs 30 images per class but co-occurrence broke the ratio (a *person* image often also contains a *bicycle*). The new sampler:
1. fills minority classes first (rarest first), skipping images that would push another minority class over its target;
2. fills majority classes only from images that contain **no** minority class (largest class last);
3. audits the realised image- and object-level counts.

The subset is fixed by `data_seed`; training seeds vary only initialization and data order.

In [ ]:
def build_imbalanced_subset(records, majority, minority, maj_n, min_n, seed):
    rng = random.Random(seed)
    minority_set, targets = set(minority), set(majority) | set(minority)
    by_class = {c: [i for i, r in enumerate(records) if c in r['labels']] for c in targets}
    selected, chosen, counts = [], set(), Counter()

    def add(i):
        selected.append(i); chosen.add(i)
        counts.update(set(records[i]['labels']) & targets)

    for c in sorted(minority, key=lambda c: len(by_class[c])):          # 1) minority first
        pool = [i for i in by_class[c] if i not in chosen]
        rng.shuffle(pool)
        for i in pool:
            if counts[c] >= min_n:
                break
            others = (set(records[i]['labels']) & minority_set) - {c}
            if any(counts[o] >= min_n for o in others):
                continue
            add(i)
    for c in sorted(majority, key=lambda c: len(by_class[c])):          # 2) majority, minority-free images only
        pool = [i for i in by_class[c] if i not in chosen and not (set(records[i]['labels']) & minority_set)]
        rng.shuffle(pool)
        for i in pool:
            if counts[c] >= maj_n:
                break
            add(i)
    return sorted(selected)


train_idx = build_imbalanced_subset(voc_train_all, MAJORITY, MINORITY, CFG['majority_n'], CFG['minority_n'],
                                    CFG['data_seed'])
train_records = [voc_train_all[i] for i in train_idx]


def class_audit(records):
    img_cnt = Counter(c for r in records for c in set(r['labels']) if c in CIDX)
    obj_cnt = Counter(o[0] for r in records for o in r['objects'] if o[0] in CIDX)
    return img_cnt, obj_cnt


pool_cnt, _ = class_audit(voc_train_all)
img_cnt, obj_cnt = class_audit(train_records)
audit = pd.DataFrame({
    'group': ['majority'] * len(MAJORITY) + ['minority'] * len(MINORITY),
    'VOC train pool (images)': [pool_cnt[c] for c in CLASSES],
    'target (images)': [CFG['majority_n']] * len(MAJORITY) + [CFG['minority_n']] * len(MINORITY),
    'realised (images)': [img_cnt[c] for c in CLASSES],
    'realised (objects)': [obj_cnt[c] for c in CLASSES],
}, index=CLASSES)
audit.to_csv(RES / 'class_audit.csv')
ratio = max(img_cnt[c] for c in CLASSES) / max(1, min(img_cnt[c] for c in CLASSES))
mean_ratio = np.mean([img_cnt[c] for c in MAJORITY]) / max(1, np.mean([img_cnt[c] for c in MINORITY]))
print(f'Imbalanced training subset: {len(train_records)} images')
print(f'Imbalance (max/min image count): {ratio:.1f}  |  majority-mean / minority-mean: {mean_ratio:.1f}')
print(audit.to_string())

fig, ax = plt.subplots(figsize=(7.5, 3.6))
order = audit.sort_values('realised (images)', ascending=True)
colors = [SERIES[0] if g == 'majority' else SERIES[1] for g in order['group']]
ax.barh(order.index, order['realised (images)'], color=colors, height=0.6)
ax.set_xlabel('Training images containing the class')
for y, v in enumerate(order['realised (images)']):
    ax.text(v + order['realised (images)'].max() * 0.01, y, str(v), va='center', color=INK2, fontsize=9)
ax.grid(axis='y', visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=SERIES[0], label='majority'), Patch(color=SERIES[1], label='minority')],
          loc='lower right')
ax.set_title('Realised class counts in the imbalanced training subset', loc='left', color=INK)
savefig(fig, 'fig_class_audit.png')

## 4. Tuning / test split
The official VOC 2012 test annotations are not public, so VOC val is split once (fixed seed) into a **tuning** set (model selection, F1 thresholds) and a **held-out test** set. Degraded variants are generated from each image inside its own split, so no image leaks across splits. The real-world datasets are never used for training or tuning.

In [ ]:
_rng = random.Random(CFG['data_seed'])
_perm = list(range(len(voc_val_all)))
_rng.shuffle(_perm)
tune_records = [voc_val_all[i] for i in _perm[:CFG['n_tune']]]
test_records = [voc_val_all[i] for i in _perm[CFG['n_tune']:]]
syn_records = test_records[:CFG['syn_eval_n']]          # fixed subset for the synthetic severity sweep
print(f'Tuning: {len(tune_records)} | Test: {len(test_records)} | Synthetic-sweep subset: {len(syn_records)}')

## 5. Real extreme-condition test sets
| Dataset | Condition | Classes used here | Split used |
|---|---|---|---|
| RTTS (RESIDE-β) | real haze | person, car, bicycle, motorbike, bus | ~15% unlabeled pool for BN adaptation, rest test (or all test if URHI is provided) |
| ExDark | real low light (10 lighting types) | all 10 (Cup has no VOC class; Table → diningtable is outside the 10) | official test split (2,563); official train/val images as the unlabeled adaptation pool |
| DAWN | real fog, rain, snow, sandstorm | the vehicle/person classes it annotates | ~15% unlabeled pool, rest test, stratified by condition |

A dataset only contributes AP for classes it actually annotates (`class mask`), so missing annotations are never counted as errors. Labels in the adaptation pools are never used.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
EXDARK_GDRIVE = {'images': '1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx', 'annotations': '1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i'}
EXDARK_CLASSLIST_URL = ('https://raw.githubusercontent.com/cs-chan/Exclusively-Dark-Image-Dataset/'
                        'master/Groundtruth/imageclasslist.txt')
EXDARK_LIGHT = {1: 'low', 2: 'ambient', 3: 'object', 4: 'single', 5: 'weak', 6: 'strong', 7: 'screen',
                8: 'window', 9: 'shadow', 10: 'twilight'}


def extract_all(src, dst):
    """Extract a zip/tar (and any archives nested inside it) into dst. A folder is used in place."""
    src, dst = Path(src), Path(dst)
    if src.is_dir():
        return src
    marker = dst / '.extracted'
    if marker.exists():
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    archives = [src]
    while archives:
        a = archives.pop()
        out = dst if a == src else a.parent / a.stem
        if zipfile.is_zipfile(a):
            with zipfile.ZipFile(a) as z:
                z.extractall(out)
        elif tarfile.is_tarfile(a):
            with tarfile.open(a) as t:
                t.extractall(out)
        else:
            print('Cannot extract', a, '- extract it manually into', out)
            continue
        if a != src:
            a.unlink()
        archives += [p for p in out.rglob('*') if p.suffix.lower() in ('.zip', '.tar') and p.is_file()]
    if list(dst.rglob('*.rar')):
        print('Found .rar archives under', dst, '- extract them manually (e.g. `unrar x`).')
    marker.touch()
    return dst


def _junk(p):
    return '__MACOSX' in p.parts or p.name.startswith('._')


def image_index(root):
    idx = defaultdict(list)
    for p in Path(root).rglob('*'):
        if p.suffix.lower() in IMG_EXT and p.is_file() and not _junk(p):
            idx[p.stem.lower()].append(p)
    return idx


def closest(paths, ref):
    """Among candidate images with the same stem, pick the one sharing the longest path prefix with ref."""
    ref_parts = Path(ref).parts
    def shared(p):
        n = 0
        for a, b in zip(Path(p).parts, ref_parts):
            if a != b:
                break
            n += 1
        return n
    return max(paths, key=shared)


def condition_from_path(path):
    """Weather condition from folder/file names, matched on word starts ('train' is not 'rain')."""
    tokens = re.split(r'[^a-z]+', str(path).lower())
    for key, cond in [('fog', 'fog'), ('haze', 'fog'), ('hazy', 'fog'), ('mist', 'fog'), ('rain', 'rain'),
                      ('snow', 'snow'), ('sand', 'sand'), ('dust', 'sand')]:
        if any(t.startswith(key) for t in tokens):
            return cond
    return 'unknown'


def parse_voc_folder(root, source):
    imgs = image_index(root)
    recs, missing = [], 0
    for x in sorted(Path(root).rglob('*.xml')):
        if _junk(x):
            continue
        try:
            fn, w, h, objs = parse_voc_xml(x)
        except ET.ParseError:
            continue
        cands = imgs.get(x.stem.lower()) or imgs.get(Path(fn).stem.lower())
        if not cands:
            missing += 1
            continue
        ip = closest(cands, x)
        recs.append(make_record(ip, objs, source, {'condition': condition_from_path(ip.relative_to(root))}, w, h))
    if missing:
        print(f'{source}: {missing} annotation files without a matching image')
    return recs


def get_exdark():
    root = DATA / 'exdark'
    try:
        if CFG['exdark_src']:
            root = extract_all(CFG['exdark_src'], root)
        elif not (root / '.done').exists():
            import gdown
            for key, fid in EXDARK_GDRIVE.items():
                out = DATA / f'exdark_{key}.zip'
                if not out.exists():
                    gdown.download(id=fid, output=str(out), quiet=False)
                extract_all(out, root / key)
            (root / '.done').touch()
    except Exception as e:
        print('ExDark download failed:', e, '\nDownload the images and groundtruth zips from '
              'github.com/cs-chan/Exclusively-Dark-Image-Dataset and set CFG["exdark_src"].')
        return None
    cl = root / 'imageclasslist.txt'
    if not cl.exists():
        try:
            urllib.request.urlretrieve(EXDARK_CLASSLIST_URL, cl)
        except Exception as e:
            print('Could not fetch ExDark imageclasslist.txt (%s); a random split will be used.' % e)
    return root


def parse_exdark(root):
    imgs, anns = {}, {}
    for p in Path(root).rglob('*'):
        if not p.is_file() or _junk(p):
            continue
        s = p.suffix.lower()
        if s in IMG_EXT:
            imgs[p.name.lower()] = p
        elif s == '.txt' and Path(p.stem).suffix.lower() in IMG_EXT:
            anns[p.stem.lower()] = p
    info = {}
    cl = next(Path(root).rglob('imageclasslist.txt'), None)
    if cl is not None:
        for line in cl.read_text(errors='ignore').splitlines()[1:]:
            parts = line.split()
            if len(parts) >= 5:
                info[parts[0].lower()] = dict(light=EXDARK_LIGHT.get(int(parts[2]), 'unknown'),
                                              indoor=int(parts[3]) == 1, split=int(parts[4]))
    recs, missing = [], 0
    for name, ap in sorted(anns.items()):
        ip = imgs.get(name)
        if ip is None:
            missing += 1
            continue
        objs = []
        for line in ap.read_text(errors='ignore').splitlines():
            parts = line.split()
            if not parts or parts[0].startswith('%') or len(parts) < 5:
                continue
            try:
                l, t, w, h = map(float, parts[1:5])
            except ValueError:
                continue
            objs.append(dict(name=parts[0], box=(l, t, l + w, t + h)))
        recs.append(make_record(ip, objs, 'exdark', info.get(name, {})))
    if missing:
        print(f'exdark: {missing} annotation files without a matching image')
    return recs


def split_real(recs, name, by=None):
    """Return (adapt_pool, test) with a fixed seed. Adaptation images are used without labels."""
    rng = random.Random(CFG['data_seed'] + 17)
    if name == 'exdark' and recs and 'split' in recs[0]['meta']:
        test = [r for r in recs if r['meta'].get('split') == 3]
        pool = [r for r in recs if r['meta'].get('split') in (1, 2)]
        rng.shuffle(pool)
        adapt = pool[:CFG['real_adapt_n']]
    else:
        groups = defaultdict(list)
        for r in recs:
            groups[r['meta'].get(by, 'all') if by else 'all'].append(r)
        adapt, test = [], []
        for g in sorted(groups):
            items = groups[g][:]
            rng.shuffle(items)
            k = int(round(len(items) * CFG['real_adapt_frac']))
            adapt += items[:k]; test += items[k:]
        rng.shuffle(adapt)
        adapt = adapt[:CFG['real_adapt_n']]
    if CFG['max_real_test']:
        rng.shuffle(test)
        test = test[:CFG['max_real_test']]
    return adapt, test


def class_mask(recs):
    present = {c for r in recs for c in r['labels'] if c in CIDX}
    return np.array([c in present for c in CLASSES])


REAL = {}   # name -> dict(test, adapt, mask, group_key)

_ex = get_exdark()
if _ex is not None:
    recs = parse_exdark(_ex)
    if recs:
        a, t = split_real(recs, 'exdark')
        REAL['exdark'] = dict(test=t, adapt=a, mask=class_mask(recs), group_key='light')

for name, key in [('rtts', 'rtts_src'), ('dawn', 'dawn_src')]:
    src = CFG[key]
    if not src or not Path(src).exists():
        print(f'{name.upper()} not found at {src!r} - skipped. Set CFG["{key}"] to the zip or folder.')
        continue
    recs = parse_voc_folder(extract_all(src, DATA / name), name)
    if not recs:
        print(f'{name.upper()}: no VOC-XML annotations found - skipped.')
        continue
    by = 'condition' if name == 'dawn' else None
    a, t = split_real(recs, name, by=by)
    if name == 'rtts' and CFG['urhi_src'] and Path(CFG['urhi_src']).exists():
        urhi_root = extract_all(CFG['urhi_src'], DATA / 'urhi')
        a = [make_record(p[0], [], 'urhi') for p in list(image_index(urhi_root).values())[:CFG['real_adapt_n']]]
        t = recs if not CFG['max_real_test'] else t
    REAL[name] = dict(test=t, adapt=a, mask=class_mask(recs), group_key=by)

rows = []
for name, d in REAL.items():
    cnt = Counter(c for r in d['test'] for c in r['labels'] if c in CIDX)
    rows.append(dict(dataset=name, test_images=len(d['test']), adapt_images=len(d['adapt']),
                     annotated_classes=', '.join(c for c, m in zip(CLASSES, d['mask']) if m),
                     **{c: cnt[c] for c in CLASSES}))
real_summary = pd.DataFrame(rows)
real_summary.to_csv(RES / 'real_datasets_summary.csv', index=False)
print(real_summary.to_string(index=False))
for src, cnt in UNMAPPED.items():
    print(f'Unmapped labels in {src} (ignored):', dict(cnt.most_common(10)))

## 6. Synthetic degradations: randomized and spatially varying
The earlier `add_haze` used one scalar transmission for the whole image, which is just a uniform blend toward grey. Real haze, dust and low light vary across the image and come with colour casts and sensor noise. Every function below takes a continuous **severity** in [1, 3] and a random generator, so each call draws new parameters:

* **haze**: atmospheric-scattering model with a *spatially varying* pseudo-depth (random horizon and tilt, near-to-far gradient) and fractal (multi-octave) density, random airlight brightness and tint, mild blur and noise.
* **lowlight**: a simplified camera pipeline: linearize (gamma 2.2), cut exposure, random white-balance shift, Poisson shot noise + Gaussian read noise, re-apply gamma.
* **snow**: desaturation and whitening veil plus two layers of motion-blurred flakes (near/large, far/small) at a random angle.
* **rain**: motion-blurred streaks at a random angle, darkening, blur and a light veil.
* **sandstorm**: depth- and fractal-modulated dust scattering with an orange-brown airlight, blue-channel absorption and dust grains.

Roles: during training, a random type and a continuous severity are drawn per image (augmentation). For evaluation, fixed severities 1/2/3 with a per-image seed give a reproducible severity sweep. Real datasets remain the main test of robustness.

In [ ]:
def _lerp(sev, lo, hi):
    a = (float(np.clip(sev, 1, 3)) - 1) / 2
    return lo + a * (hi - lo)


def fractal_noise(h, w, rng, octaves=5, base=3):
    """Multi-octave value noise in [0, 1]: smooth, patchy structure at several scales."""
    out = np.zeros((h, w), np.float32)
    amp, total = 1.0, 0.0
    for o in range(octaves):
        cells = base * 2 ** o
        gh, gw = max(2, int(round(cells * h / max(h, w))) + 1), max(2, int(round(cells * w / max(h, w))) + 1)
        grid = rng.random((gh, gw)).astype(np.float32)
        out += amp * cv2.resize(grid, (w, h), interpolation=cv2.INTER_CUBIC)
        total += amp
        amp *= 0.5
    out /= total
    return (out - out.min()) / (out.max() - out.min() + 1e-6)


def pseudo_depth(h, w, rng):
    """Rough scene depth in [0, 1] (1 = far): far above a random tilted horizon, nearer toward the bottom."""
    y = np.linspace(0, 1, h, dtype=np.float32)[:, None]
    x = np.linspace(-1, 1, w, dtype=np.float32)[None, :]
    horizon = rng.uniform(0.25, 0.6) + rng.uniform(-0.12, 0.12) * x
    d = 1.0 - 0.8 * np.clip((y - horizon) / (1.0 - horizon + 1e-6), 0, 1)
    return np.clip(0.75 * d + 0.25 * fractal_noise(h, w, rng, octaves=3), 0, 1)


def _motion_kernel(length, angle_deg):
    """Line kernel of the given length, rotated by angle_deg from vertical."""
    length = max(3, int(length) | 1)
    k = np.zeros((length, length), np.float32)
    k[:, length // 2] = 1.0
    M = cv2.getRotationMatrix2D(((length - 1) / 2, (length - 1) / 2), angle_deg, 1.0)
    k = cv2.warpAffine(k, M, (length, length))
    return k / (k.sum() + 1e-6)


def _to_float(img):
    return img.astype(np.float32) / 255.0


def _to_uint8(x):
    return (np.clip(x, 0, 1) * 255 + 0.5).astype(np.uint8)


def _sensor(x, rng, blur_sigma, noise_std):
    if blur_sigma > 0.05:
        x = cv2.GaussianBlur(x, (0, 0), blur_sigma)
    return x + rng.normal(0, noise_std, x.shape).astype(np.float32)


def deg_haze(img, sev, rng):
    x = _to_float(img)
    h, w = x.shape[:2]
    d = pseudo_depth(h, w, rng)
    hetero = 0.7 + 0.6 * fractal_noise(h, w, rng)
    beta = _lerp(sev, 0.6, 2.8) * rng.uniform(0.85, 1.15)
    A = np.clip(rng.uniform(0.75, 0.95) + rng.uniform(-0.03, 0.03, 3), 0, 1).astype(np.float32)
    t = np.exp(-beta * d * hetero)[..., None]
    x = x * t + A * (1 - t)
    return _to_uint8(_sensor(x, rng, _lerp(sev, 0.0, 0.8), 0.008))


def deg_lowlight(img, sev, rng):
    lin = _to_float(img) ** 2.2
    lin = lin * _lerp(sev, 0.25, 0.04) * rng.uniform(0.8, 1.2)
    lin = lin * (1 + rng.uniform(-0.15, 0.15, 3)).astype(np.float32)          # white-balance shift
    photons = _lerp(sev, 300.0, 40.0)                                         # fewer photons = more shot noise
    lin = rng.poisson(np.clip(lin, 0, 1) * photons).astype(np.float32) / photons
    lin = lin + rng.normal(0, _lerp(sev, 0.002, 0.01), lin.shape).astype(np.float32)
    return _to_uint8(np.clip(lin, 0, 1) ** (1 / 2.2))


def deg_snow(img, sev, rng):
    x = _to_float(img)
    h, w = x.shape[:2]
    veil = _lerp(sev, 0.12, 0.45)
    gray = x.mean(axis=2, keepdims=True)
    x = x * (1 - 0.4 * veil) + gray * 0.4 * veil
    x = x * (1 - veil) + 0.9 * veil
    angle = rng.uniform(-30, 30)
    scale = max(h, w) / 500
    for density, size, blur_len, opacity in [(_lerp(sev, 0.004, 0.02), 1, 3, 0.75),
                                             (_lerp(sev, 0.0008, 0.005), 3, 7, 0.9)]:
        m = (rng.random((h, w)) < density).astype(np.float32)
        k = max(1, int(round(size * scale)))
        if k > 1:
            m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
        m = cv2.filter2D(m, -1, _motion_kernel(blur_len * scale, angle))
        m = cv2.GaussianBlur(m, (0, 0), 0.6)
        m = np.clip(m / (m.max() + 1e-6) * opacity * 1.5, 0, opacity)[..., None]
        x = x * (1 - m) + m
    return _to_uint8(_sensor(x, rng, _lerp(sev, 0.0, 0.6), 0.005))


def deg_rain(img, sev, rng):
    x = _to_float(img)
    h, w = x.shape[:2]
    scale = max(h, w) / 500
    drops = (rng.random((h, w)) < _lerp(sev, 0.002, 0.008)).astype(np.float32)
    streaks = cv2.filter2D(drops, -1, _motion_kernel(_lerp(sev, 12, 35) * scale, rng.uniform(-20, 20)))
    streaks = cv2.GaussianBlur(streaks, (0, 0), 0.7)
    streaks = np.clip(streaks / (streaks.max() + 1e-6), 0, 1)[..., None]
    alpha = _lerp(sev, 0.35, 0.7)
    x = x * _lerp(sev, 0.95, 0.75)
    x = x * (1 - alpha * streaks) + alpha * streaks * 0.85
    veil = _lerp(sev, 0.05, 0.2)
    x = x * (1 - veil) + veil * 0.7
    return _to_uint8(_sensor(x, rng, _lerp(sev, 0.3, 1.2), 0.006))


def deg_sandstorm(img, sev, rng):
    x = _to_float(img)
    h, w = x.shape[:2]
    dust = np.clip(np.array([0.80, 0.62, 0.40]) * rng.uniform(0.9, 1.1, 3), 0, 1).astype(np.float32)
    d = pseudo_depth(h, w, rng)
    hetero = 0.6 + 0.8 * fractal_noise(h, w, rng)
    t = np.exp(-_lerp(sev, 0.8, 3.2) * rng.uniform(0.85, 1.15) * d * hetero)[..., None]
    x = x * t + dust * (1 - t)
    x[..., 2] *= _lerp(sev, 0.9, 0.6)                                           # blue absorption, yellow cast
    grains = cv2.GaussianBlur((rng.random((h, w)) < _lerp(sev, 0.002, 0.01)).astype(np.float32), (0, 0), 0.8)
    grains = np.clip(grains / (grains.max() + 1e-6), 0, 1)[..., None] * 0.5
    x = x * (1 - grains) + dust * grains
    return _to_uint8(_sensor(x, rng, _lerp(sev, 0.3, 1.0), 0.008))


DEGRADATIONS = {'haze': deg_haze, 'lowlight': deg_lowlight, 'snow': deg_snow, 'rain': deg_rain,
                'sandstorm': deg_sandstorm}
DEG_ID = {k: i + 1 for i, k in enumerate(DEGRADATIONS)}


def apply_degradation(img, kind, sev, rng):
    if kind in (None, 'none', 'clean'):
        return img
    return DEGRADATIONS[kind](img, sev, rng)


def random_degradation(img, rng, p=1.0, kinds=None):
    """Training augmentation: with probability p, a random type at a random continuous severity in [1, 3]."""
    if rng.random() >= p:
        return img
    kinds = kinds or CFG['degr_types']
    return apply_degradation(img, kinds[rng.integers(len(kinds))], rng.uniform(1.0, 3.0), rng)


def fixed_rng(seed, i, kind='clean', sev=0):
    """Reproducible generator for evaluation: the same image always gets the same degradation."""
    return np.random.default_rng([int(seed), int(i), DEG_ID.get(kind, 0), int(round(sev * 10))])


# Visual check: one VOC image across types and severities (for the slides)
_demo_img = np.asarray(Image.open(syn_records[0]['path']).convert('RGB'))
fig, axes = plt.subplots(3, len(DEGRADATIONS) + 1, figsize=(2.6 * (len(DEGRADATIONS) + 1), 6.2))
for row, sev in enumerate([1, 2, 3]):
    axes[row, 0].imshow(_demo_img); axes[row, 0].set_ylabel(f'severity {sev}', color=INK2)
    for col, kind in enumerate(DEGRADATIONS, start=1):
        axes[row, col].imshow(apply_degradation(_demo_img, kind, sev, fixed_rng(0, 0, kind, sev)))
        if row == 0:
            axes[row, col].set_title(kind, color=INK)
    if row == 0:
        axes[row, 0].set_title('clean', color=INK)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
savefig(fig, 'fig_degradations.png')

**bold text**## 7. Datasets, loaders and cached evaluation banks
* **Training** images are resized, randomly cropped and flipped; in augmentation configs a random degradation is applied first (on the full-resolution image). The augmentation RNG is seeded by *(seed, epoch, index)*, so runs are reproducible yet every epoch sees new degradations.
* **Evaluation banks**: every evaluation set (clean, each synthetic type × severity, each real dataset) is rendered once to a 224×224 uint8 array on disk. All models are then scored on identical pixels, and evaluation runs at GPU speed instead of re-decoding and re-degrading images for every model.

In [ ]:
S = CFG['img_size']
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
RESIZE = int(round(S * 256 / 224))


def load_rgb(path, max_side=None):
    try:
        im = Image.open(path)
        if max_side and im.format == 'JPEG':
            im.draft('RGB', (max_side, max_side))   # fast DCT-domain downscale for large JPEGs
        im = im.convert('RGB')
    except Exception as e:                          # unreadable file: keep the run alive, but say so
        print(f'WARNING: could not read {path} ({e}); using a grey image')
        return Image.new('RGB', (S, S), (128, 128, 128))
    if max_side and max(im.size) > max_side:
        im.thumbnail((max_side, max_side), Image.BILINEAR)
    return im


def label_vector(rec):
    y = np.zeros(NC, np.float32)
    for c in rec['labels']:
        if c in CIDX:
            y[CIDX[c]] = 1.0
    return y


train_tf = T.Compose([T.Resize((RESIZE, RESIZE)), T.RandomCrop(S), T.RandomHorizontalFlip(), T.ToTensor(),
                      T.Normalize(MEAN.flatten().tolist(), STD.flatten().tolist())])
bank_tf = T.Compose([T.Resize((S, S)), T.PILToTensor()])
_norm = T.Normalize(MEAN.flatten().tolist(), STD.flatten().tolist())


def paired_tf(a, b):
    """Same resize/crop/flip for a clean and a degraded view (consistency training)."""
    a, b = TF.resize(a, [RESIZE, RESIZE]), TF.resize(b, [RESIZE, RESIZE])
    i, j, h, w = T.RandomCrop.get_params(a, (S, S))
    a, b = TF.crop(a, i, j, h, w), TF.crop(b, i, j, h, w)
    if random.random() < 0.5:
        a, b = TF.hflip(a), TF.hflip(b)
    return _norm(TF.to_tensor(a)), _norm(TF.to_tensor(b))


class MultiLabelDataset(Dataset):
    """mode='train' -> normalized augmented tensor; mode='bank' -> uint8 224x224 tensor.
    degrade: None | 'random' | (kind, severity)."""

    def __init__(self, records, mode='train', degrade=None, aug_prob=1.0, seed=0, paired=False):
        self.records, self.mode, self.degrade = records, mode, degrade
        self.aug_prob, self.seed, self.paired, self.epoch = aug_prob, seed, paired, 0

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        img = load_rgb(rec['path'], CFG['max_side'])
        y = torch.from_numpy(label_vector(rec))
        deg = None
        if self.degrade == 'random':
            rng = np.random.default_rng([self.seed, self.epoch, i])
            p = 1.0 if self.paired else self.aug_prob
            deg = Image.fromarray(random_degradation(np.asarray(img), rng, p))
        elif isinstance(self.degrade, tuple):
            kind, sev = self.degrade
            deg = Image.fromarray(apply_degradation(np.asarray(img), kind, sev, fixed_rng(self.seed, i, kind, sev)))
        if self.mode == 'bank':
            return bank_tf(deg if deg is not None else img), y
        if self.paired:
            return paired_tf(img, deg), y
        return train_tf(deg if deg is not None else img), y


BANKS = {}   # name -> dict(records, mask, group_key)


def bank_paths(name):
    return CACHE / f'{name}_X.npy', CACHE / f'{name}_Y.npy'


def build_bank(name, records, degrade=None, mask=None, group_key=None, seed=None):
    fx, fy = bank_paths(name)
    BANKS[name] = dict(records=records, mask=np.ones(NC, bool) if mask is None else mask, group_key=group_key)
    if fx.exists() and fy.exists():
        return
    seed = CFG['data_seed'] if seed is None else seed
    ds = MultiLabelDataset(records, mode='bank', degrade=degrade, seed=seed)
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=CFG['num_workers'])
    tmp = CACHE / f'{name}_X.tmp.npy'
    X = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.uint8, shape=(len(ds), 3, S, S))
    Y, k = np.zeros((len(ds), NC), np.float32), 0
    for x, y in tqdm(dl, desc=f'bank {name}', leave=False):
        X[k:k + len(x)] = x.numpy(); Y[k:k + len(x)] = y.numpy(); k += len(x)
    X.flush(); del X
    os.replace(tmp, fx)
    np.save(fy, Y)


def bank_Y(name):
    return np.load(bank_paths(name)[1])


t0 = time.time()
build_bank('voc_tune', tune_records)
build_bank('voc_test', test_records)
build_bank('voc_syn_clean', syn_records)
for kind in CFG['degr_types']:
    for sev in CFG['severities']:
        build_bank(f'voc_syn_{kind}_s{sev}', syn_records, degrade=(kind, sev))
for name, d in REAL.items():
    build_bank(f'{name}_test', d['test'], mask=d['mask'], group_key=d['group_key'])
    build_bank(f'{name}_adapt', d['adapt'])
print(f'Evaluation banks ready ({len(BANKS)} sets, {time.time() - t0:.0f}s)')

## 8. ResNet-50: architecture overview
* **Stem**: 7×7 conv (64 channels, stride 2) + BN + ReLU + 3×3 max-pool (stride 2): 224×224 → 56×56.
* **Four stages of bottleneck blocks** `[3, 4, 6, 3]`. Each block is 1×1 conv (reduce) → 3×3 conv → 1×1 conv (expand ×4), with BatchNorm after every conv and an identity shortcut: output = ReLU(F(x) + x). The first block of each stage uses a 1×1 projection shortcut; stages 2–4 halve the resolution (torchvision puts the stride in the 3×3 conv).
* **Head**: global average pooling → 2048-d vector → fully connected layer. The 1000-way ImageNet layer is replaced by a 10-way layer; a sigmoid per class gives independent presence probabilities (multi-label).
* **Size**: 1 + 16×3 + 1 = 50 weighted layers.
* **Why ResNet-50 here**: strong ImageNet features that transfer to small datasets; the standard reference model in corruption-robustness benchmarks; and its BatchNorm layers make test-time BN adaptation (Section 11) possible.

In [ ]:
def build_model(pretrained=None):
    pretrained = CFG['pretrained'] if pretrained is None else pretrained
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, NC)
    return m


def architecture_table():
    m = build_model(pretrained=False).eval()
    stages = [('stem (conv1+bn+relu+maxpool)', [m.conv1, m.bn1, m.maxpool], m.maxpool), ('stage 1 (layer1)', [m.layer1], m.layer1),
              ('stage 2 (layer2)', [m.layer2], m.layer2), ('stage 3 (layer3)', [m.layer3], m.layer3),
              ('stage 4 (layer4)', [m.layer4], m.layer4), ('global avg pool', [m.avgpool], m.avgpool), ('fc (2048→10)', [m.fc], m.fc)]
    shapes = {}
    hooks = [mod.register_forward_hook(lambda mod, i, o, n=n: shapes.__setitem__(n, tuple(o.shape[1:])))
             for n, _, mod in stages]
    with torch.no_grad():
        m(torch.zeros(1, 3, 224, 224))
    for h in hooks:
        h.remove()
    rows = []
    for n, mods, _ in stages:
        params = sum(p.numel() for mod in mods for p in mod.parameters())
        blocks = len(mods[0]) if isinstance(mods[0], nn.Sequential) else '-'
        rows.append(dict(stage=n, bottleneck_blocks=blocks, output_shape='×'.join(map(str, shapes[n])),
                         params_M=round(params / 1e6, 3)))
    df = pd.DataFrame(rows)
    total = sum(p.numel() for p in m.parameters())
    return df, total


arch_df, arch_total = architecture_table()
arch_df.to_csv(RES / 'resnet50_architecture.csv', index=False)
print(arch_df.to_string(index=False))
print(f'Total parameters with a 10-way head: {arch_total / 1e6:.2f} M')

# Diagram for the slides
fig, ax = plt.subplots(figsize=(12, 3.4))
ax.set_xlim(0, 12); ax.set_ylim(0, 3.4); ax.axis('off')
boxes = [('Input\n224×224×3', '', 0.2), ('Stem\n7×7 conv, s2\n+ max-pool', '56×56×64', 1.55),
         ('Stage 1\n3 bottlenecks', '56×56×256', 3.0), ('Stage 2\n4 bottlenecks', '28×28×512', 4.45),
         ('Stage 3\n6 bottlenecks', '14×14×1024', 5.9), ('Stage 4\n3 bottlenecks', '7×7×2048', 7.35),
         ('Global\navg pool', '2048', 8.8), ('FC 2048→10\n+ sigmoid', '10 probs', 10.25)]
for i, (label, shape, x0) in enumerate(boxes):
    ax.add_patch(plt.Rectangle((x0, 1.5), 1.25, 1.3, facecolor='#cde2fb' if 'Stage' in label else '#f0efec',
                               edgecolor=SERIES[0] if 'Stage' in label else INK2, linewidth=1.2))
    ax.text(x0 + 0.625, 2.15, label, ha='center', va='center', fontsize=8.5, color=INK)
    ax.text(x0 + 0.625, 1.3, shape, ha='center', va='top', fontsize=8, color=INK2)
    if i < len(boxes) - 1:
        ax.annotate('', xy=(boxes[i + 1][2], 2.15), xytext=(x0 + 1.25, 2.15),
                    arrowprops=dict(arrowstyle='->', color=INK2, lw=1))
ax.text(0.2, 0.55, 'Bottleneck block:  x → [1×1 conv, BN, ReLU] → [3×3 conv, BN, ReLU] → [1×1 conv (×4 channels), BN] → (+ x or 1×1 projection) → ReLU',
        fontsize=8.5, color=INK2)
ax.text(0.2, 3.2, f'ResNet-50 (ImageNet-pretrained), {arch_total / 1e6:.1f} M parameters with the 10-class head',
        fontsize=10, color=INK)
savefig(fig, 'fig_resnet50.png')

## 9. Loss: multi-label BCE with class-balanced positive weights
The baseline loss is `BCEWithLogitsLoss` (one sigmoid per class). For the *weighting* configurations, each class's positive term is re-weighted by the inverse **effective number of samples** (Cui et al., CVPR 2019): $E_c = (1-\beta^{n_c})/(1-\beta)$ with $\beta = 0.999$ and $n_c$ the number of training images containing class $c$. Weights are normalized so the most frequent class has weight 1, and capped at `max_pos_weight`. This raises the gradient from scarce positive examples without changing the negative terms.

In [ ]:
def class_pos_counts(records):
    return np.sum([label_vector(r) for r in records], axis=0)


def make_criterion(weighting, records):
    pos = class_pos_counts(records)
    n = len(records)
    if weighting in (None, 'none', False):
        return nn.BCEWithLogitsLoss(), np.ones(NC, np.float32)
    if weighting == 'cb_pos':
        beta = CFG['cb_beta']
        eff = (1 - beta ** np.maximum(pos, 1)) / (1 - beta)
        w = 1.0 / eff
        pw = w / w.min()
    elif weighting == 'pos_weight':
        pw = (n - pos) / np.maximum(pos, 1)
    else:
        raise ValueError(weighting)
    pw = np.clip(pw, 1.0, CFG['max_pos_weight']).astype(np.float32)
    return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=device)), pw


_, _pw = make_criterion(CFG['weighting'], train_records)
print(pd.DataFrame({'train images with class': class_pos_counts(train_records).astype(int),
                    'positive weight': _pw.round(2)}, index=CLASSES))

## 10. Training loop
AdamW (lr 1e-4, weight decay 1e-4), 1 warm-up epoch then cosine decay, batch size 32, mixed precision on GPU. After every epoch the model is scored on the **clean tuning set** (mAP); the best epoch is kept. Model selection never sees degraded or real-condition data.

Four configurations form the ablation (all on the same imbalanced subset and the same test sets):

| Config | Class-balanced weights | Degradation augmentation |
|---|---|---|
| baseline | no | no |
| weighting | yes | no |
| augmentation | no | yes |
| combined | yes | yes |

Test-time BN adaptation (Section 11) is applied on top of each trained model at evaluation, so it adds rows without extra training.

In [ ]:
CONFIGS = {
    'baseline':     dict(weighting=False, augment=False),
    'weighting':    dict(weighting=True,  augment=False),
    'augmentation': dict(weighting=False, augment=True),
    'combined':     dict(weighting=True,  augment=True),
}


@torch.no_grad()
def predict_bank(model, name, bs=128):
    X = np.load(bank_paths(name)[0], mmap_mode='r')
    model.eval()
    mean, std = MEAN.to(device), STD.to(device)
    out = []
    for s in range(0, len(X), bs):
        x = torch.from_numpy(np.array(X[s:s + bs])).to(device).float().div_(255).sub_(mean).div_(std)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
            out.append(torch.sigmoid(model(x).float()).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, NC), np.float32)


def mean_ap(probs, Y, mask=None):
    aps = [average_precision_score(Y[:, c], probs[:, c]) for c in range(NC)
           if (mask is None or mask[c]) and 0 < Y[:, c].sum() < len(Y)]
    return float(np.mean(aps)) if aps else float('nan')


def binary_js(lc, ld, w):
    """Class-weighted Jensen-Shannon divergence between Bernoulli predictions of two views."""
    pc, pd_ = torch.sigmoid(lc.float()).clamp(1e-6, 1 - 1e-6), torch.sigmoid(ld.float()).clamp(1e-6, 1 - 1e-6)
    m = 0.5 * (pc + pd_)
    def kl(p, q):
        return p * torch.log(p / q) + (1 - p) * torch.log((1 - p) / (1 - q))
    return ((0.5 * kl(pc, m) + 0.5 * kl(pd_, m)) * w).mean()


def ckpt_path(name, seed):
    return CKPT / f'cls_{name}_s{seed}.pt'


def load_classifier(name, seed):
    m = build_model(pretrained=False)
    m.load_state_dict(torch.load(ckpt_path(name, seed), map_location='cpu')['state_dict'])
    return m.to(device).eval()


def train_classifier(name, spec, seed):
    if ckpt_path(name, seed).exists():
        return load_classifier(name, seed)
    set_seed(seed)
    consistency = spec.get('consistency', 0.0)
    ds = MultiLabelDataset(train_records, mode='train', degrade='random' if spec['augment'] else None,
                           aug_prob=CFG['aug_prob'], seed=seed, paired=consistency > 0)
    dl = DataLoader(ds, batch_size=CFG['batch_size'], shuffle=True, drop_last=True, num_workers=CFG['num_workers'],
                    pin_memory=AMP, generator=torch.Generator().manual_seed(seed))
    model = build_model().to(device)
    crit, pw = make_criterion(CFG['weighting'] if spec['weighting'] else 'none', train_records)
    cls_w = torch.tensor(pw / pw.mean(), device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    total, warm = CFG['epochs'] * len(dl), max(1, CFG['warmup_epochs'] * len(dl))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    Y_tune = bank_Y('voc_tune')
    best, best_state, history = -1.0, None, []
    for ep in range(CFG['epochs']):
        ds.epoch = ep
        model.train()
        t0, run_loss, nb = time.time(), 0.0, 0
        for xb, y in dl:
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
                if consistency > 0:
                    xc, xd = xb[0].to(device, non_blocking=True), xb[1].to(device, non_blocking=True)
                    lc, ld = model(torch.cat([xc, xd])).chunk(2)
                    loss = 0.5 * (crit(lc.float(), y) + crit(ld.float(), y)) + consistency * binary_js(lc, ld, cls_w)
                else:
                    loss = crit(model(xb.to(device, non_blocking=True)).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item(); nb += 1
        tune_map = mean_ap(predict_bank(model, 'voc_tune'), Y_tune)
        history.append(dict(epoch=ep + 1, loss=run_loss / max(nb, 1), tune_mAP=tune_map, seconds=time.time() - t0))
        print(f'[{name} s{seed}] epoch {ep + 1}/{CFG["epochs"]}  loss {run_loss / max(nb, 1):.4f}  '
              f'tune mAP {tune_map:.4f}  ({time.time() - t0:.0f}s)')
        if tune_map > best:
            best = tune_map
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save(dict(state_dict=best_state, history=history, spec=spec, seed=seed, classes=CLASSES,
                    pos_weight=pw.tolist(), cfg=CFG), ckpt_path(name, seed))
    model.load_state_dict(best_state)
    return model.eval()

## 11. Metrics and test-time BatchNorm adaptation
* **Per-class AP and mAP** (threshold-free; the standard VOC classification metric) are the main numbers. Results are also split into **majority-mean** and **minority-mean** AP.
* **F1** uses per-class thresholds tuned on the clean tuning set (a fixed 0.5 threshold hurts rare classes most). The same thresholds are reused on every test set.
* **BN adaptation** (Schneider et al., NeurIPS 2020): BatchNorm running statistics are re-estimated on *unlabeled* images from the target condition (the adaptation pool of each real dataset), then the model is evaluated on that dataset's test images. No labels and no gradient steps are used.

In [ ]:
def tune_thresholds(probs, Y):
    grid = np.linspace(0.05, 0.95, 19)
    th = np.full(NC, 0.5, np.float32)
    for c in range(NC):
        if Y[:, c].sum() == 0:
            continue
        f1s = []
        for t in grid:
            p = probs[:, c] >= t
            tp = (p & (Y[:, c] == 1)).sum()
            f1s.append(2 * tp / max(1, p.sum() + Y[:, c].sum()))
        th[c] = grid[int(np.argmax(f1s))]
    return th


def per_class_metrics(probs, Y, mask, th):
    out = {}
    for c in range(NC):
        n_pos = Y[:, c].sum()
        if not mask[c] or n_pos == 0 or n_pos == len(Y):
            continue
        p = probs[:, c] >= th[c]
        tp = (p & (Y[:, c] == 1)).sum()
        out[CLASSES[c]] = dict(AP=float(average_precision_score(Y[:, c], probs[:, c])),
                               F1=float(2 * tp / max(1, p.sum() + n_pos)), n_pos=int(n_pos))
    return out


def summary_rows(pc, base):
    rows = [dict(base, cls=c, **v) for c, v in pc.items()]
    for tag, names in [('ALL', CLASSES), ('MAJ', MAJORITY), ('MIN', MINORITY)]:
        vals = [pc[c] for c in names if c in pc]
        if vals:
            rows.append(dict(base, cls=tag, AP=float(np.mean([v['AP'] for v in vals])),
                             F1=float(np.mean([v['F1'] for v in vals])), n_pos=len(vals)))
    return rows


def evaluate_bank(model, name, th, config, seed, bn_adapt=False, save_pred=True):
    probs = predict_bank(model, name)
    Y, info = bank_Y(name), BANKS[name]
    if save_pred:
        np.save(PRED / f'{config}_s{seed}_{name}{"_bn" if bn_adapt else ""}.npy', probs.astype(np.float16))
    base = dict(config=config, seed=seed, eval_set=name, bn_adapt=bn_adapt)
    rows = summary_rows(per_class_metrics(probs, Y, info['mask'], th), dict(base, group='all'))
    if info['group_key']:
        groups = np.array([r['meta'].get(info['group_key'], 'unknown') for r in info['records']])
        for g in sorted(set(groups)):
            idx = groups == g
            if idx.sum() >= 20:
                rows += summary_rows(per_class_metrics(probs[idx], Y[idx], info['mask'], th), dict(base, group=g))
    return rows


@torch.no_grad()
def bn_adapt(model, adapt_bank, bs=None, source_weight=0.0):
    """Re-estimate BatchNorm statistics on unlabeled target images (optionally mixed with source stats)."""
    bs = bs or CFG['bn_adapt_batch']
    m = copy.deepcopy(model)
    src = {n: (b.running_mean.clone(), b.running_var.clone()) for n, b in m.named_modules()
           if isinstance(b, nn.BatchNorm2d)}
    for b in m.modules():
        if isinstance(b, nn.BatchNorm2d):
            b.reset_running_stats(); b.momentum = None      # cumulative average over all adaptation batches
    m.train()
    X = np.load(bank_paths(adapt_bank)[0], mmap_mode='r')
    mean, std = MEAN.to(device), STD.to(device)
    for s in range(0, len(X), bs):
        x = torch.from_numpy(np.array(X[s:s + bs])).to(device).float().div_(255).sub_(mean).div_(std)
        if len(x) > 1:
            m(x)
    if source_weight > 0:
        for n, b in m.named_modules():
            if n in src:
                b.running_mean.mul_(1 - source_weight).add_(src[n][0] * source_weight)
                b.running_var.mul_(1 - source_weight).add_(src[n][1] * source_weight)
    return m.eval()

## 12. Run the classification ablation
Each configuration is trained with every seed in `CFG['seeds']`. Checkpoints are saved per run; re-running this cell skips finished runs.

In [ ]:
RUN_NOVELTY_PREVIEW = False   # Section 19; keep False for the Stage-1 results
ALL_CONFIGS = dict(CONFIGS)
if RUN_NOVELTY_PREVIEW:
    ALL_CONFIGS['proposed'] = dict(weighting=True, augment=True, consistency=CFG['consistency_weight'])

t_all = time.time()
for name, spec in ALL_CONFIGS.items():
    for seed in CFG['seeds']:
        if not ckpt_path(name, seed).exists():
            train_classifier(name, spec, seed)
            if AMP:
                torch.cuda.empty_cache()
print(f'All classification runs ready ({(time.time() - t_all) / 60:.1f} min)')

## 13. Evaluate every model on every test set
For each (config, seed): thresholds are tuned on the clean tuning set; the model is scored on clean VOC test, the synthetic severity sweep and each real dataset; then BN-adapted copies are scored on each real dataset. Results are appended to `results/cls_results.csv` after each model.

In [ ]:
CLS_CSV = RES / 'cls_results.csv'
cls_df = pd.read_csv(CLS_CSV) if CLS_CSV.exists() else pd.DataFrame()
done = set(zip(cls_df['config'], cls_df['seed'])) if len(cls_df) else set()
EVAL_SETS = ['voc_test', 'voc_syn_clean'] + [n for n in BANKS if n.startswith('voc_syn_') and n != 'voc_syn_clean'] \
            + [f'{r}_test' for r in REAL]

for name in ALL_CONFIGS:
    for seed in CFG['seeds']:
        if (name, seed) in done:
            continue
        model = load_classifier(name, seed)
        th = tune_thresholds(predict_bank(model, 'voc_tune'), bank_Y('voc_tune'))
        rows = []
        for ev in EVAL_SETS:
            rows += evaluate_bank(model, ev, th, name, seed)
        for r in REAL:
            rows += evaluate_bank(bn_adapt(model, f'{r}_adapt'), f'{r}_test', th, name, seed, bn_adapt=True)
        cls_df = pd.concat([cls_df, pd.DataFrame(rows)], ignore_index=True)
        cls_df.to_csv(CLS_CSV, index=False)
        print(f'evaluated {name} s{seed}')
print(f'{len(cls_df)} result rows in {CLS_CSV}')

## 14. Classification results: tables and plots
Every number is the mean ± std over training seeds. `syn avg` averages all synthetic types and severities. `+BN` rows apply test-time BN adaptation (real datasets only).

In [ ]:
def fmt(m, s):
    return '–' if pd.isna(m) else (f'{m:.3f}' if pd.isna(s) or s == 0 else f'{m:.3f} ± {s:.3f}')


def results_table(df, cls_tag='ALL', metric='AP'):
    d = df[(df['cls'] == cls_tag) & (df['group'] == 'all')].copy()
    d['col'] = d['eval_set']
    syn = d['eval_set'].str.match(r'voc_syn_.+_s\d')
    d.loc[syn, 'col'] = 'syn avg'
    per_seed = d.groupby(['config', 'bn_adapt', 'seed', 'col'])[metric].mean().reset_index()
    agg = per_seed.groupby(['config', 'bn_adapt', 'col'])[metric].agg(['mean', 'std']).reset_index()
    cols = [c for c in ['voc_test', 'syn avg'] + [f'{r}_test' for r in REAL] if c in set(agg['col'])]
    rows = []
    for config in ALL_CONFIGS:
        for bn in [False, True]:
            sub = agg[(agg['config'] == config) & (agg['bn_adapt'] == bn)]
            if sub.empty:
                continue
            row = {'config': config + (' +BN' if bn else '')}
            for c in cols:
                r = sub[sub['col'] == c]
                row[c] = fmt(r['mean'].iloc[0], r['std'].iloc[0]) if len(r) else '–'
            rows.append(row)
    return pd.DataFrame(rows).set_index('config')


def to_markdown(df):
    cols = [df.index.name or ''] + list(df.columns)
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for idx, row in df.iterrows():
        lines.append('| ' + ' | '.join([str(idx)] + [str(v) for v in row.values]) + ' |')
    return '\n'.join(lines)


TABLES = {}
for slug, tag, metric, title in [('map_all', 'ALL', 'AP', 'mAP, all 10 classes'),
                                 ('map_minority', 'MIN', 'AP', 'mAP, minority classes'),
                                 ('map_majority', 'MAJ', 'AP', 'mAP, majority classes'),
                                 ('f1_all', 'ALL', 'F1', 'macro-F1 (tuned thresholds), all classes')]:
    TABLES[title] = results_table(cls_df, tag, metric)
    TABLES[title].to_csv(RES / f'cls_table_{slug}.csv')
    print(f'\n=== {title} ===')
    print(TABLES[title].to_string())

In [ ]:
# Severity curves: mAP vs severity for each synthetic degradation (0 = the same images, clean)
def sev_curve(df, kind, config, tag):
    d = df[(df['config'] == config) & (df['cls'] == tag) & (df['group'] == 'all') & (~df['bn_adapt'].astype(bool))]
    xs, ms, ss = [], [], []
    for sev in [0] + list(CFG['severities']):
        ev = 'voc_syn_clean' if sev == 0 else f'voc_syn_{kind}_s{sev}'
        v = d[d['eval_set'] == ev].groupby('seed')['AP'].mean()
        if len(v):
            xs.append(sev); ms.append(v.mean()); ss.append(v.std() if len(v) > 1 else 0)
    return np.array(xs), np.array(ms), np.array(ss)


kinds = CFG['degr_types']
fig, axes = plt.subplots(2, len(kinds), figsize=(3.1 * len(kinds), 5.8), sharey='row')
axes = np.atleast_2d(axes)
for row, (tag, ylabel) in enumerate([('ALL', 'mAP (all classes)'), ('MIN', 'mAP (minority)')]):
    for col, kind in enumerate(kinds):
        ax = axes[row, col]
        for k, config in enumerate(ALL_CONFIGS):
            x, m, s = sev_curve(cls_df, kind, config, tag)
            if len(x) == 0:
                continue
            ax.plot(x, m, '-o', color=SERIES[k], lw=2, ms=5, label=config)
            ax.fill_between(x, m - s, m + s, color=SERIES[k], alpha=0.12, lw=0)
        ax.set_xticks([0] + list(CFG['severities']))
        if row == 0:
            ax.set_title(kind, color=INK)
        if row == 1:
            ax.set_xlabel('severity (0 = clean)')
        if col == 0:
            ax.set_ylabel(ylabel)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.suptitle('Synthetic severity sweep on VOC test (mean ± std over seeds)', x=0.01, ha='left', color=INK)
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.legend(handles, labels, loc='lower center', ncol=len(labels), fontsize=9)
savefig(fig, 'fig_severity_curves.png')

In [ ]:
# Real-world results: grouped bars (solid = no adaptation, hatched = +BN adaptation)
def real_bar_data(df, tag):
    d = df[(df['cls'] == tag) & (df['group'] == 'all') & df['eval_set'].isin([f'{r}_test' for r in REAL])]
    return d.groupby(['eval_set', 'config', 'bn_adapt', 'seed'])['AP'].mean().groupby(
        ['eval_set', 'config', 'bn_adapt']).agg(['mean', 'std']).reset_index()


if REAL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    for ax, (tag, title) in zip(axes, [('ALL', 'All annotated classes'), ('MIN', 'Minority classes')]):
        d = real_bar_data(cls_df, tag)
        sets = [f'{r}_test' for r in REAL]
        width = 0.8 / (2 * len(ALL_CONFIGS))
        for k, config in enumerate(ALL_CONFIGS):
            for b, bn in enumerate([False, True]):
                xs, ms, ss = [], [], []
                for i, s in enumerate(sets):
                    r = d[(d['eval_set'] == s) & (d['config'] == config) & (d['bn_adapt'] == bn)]
                    if len(r):
                        xs.append(i - 0.4 + width * (2 * k + b) + width / 2)
                        ms.append(r['mean'].iloc[0]); ss.append(0 if pd.isna(r['std'].iloc[0]) else r['std'].iloc[0])
                ax.bar(xs, ms, width * 0.92, yerr=ss, color=SERIES[k], hatch='///' if bn else None,
                       edgecolor=SURFACE if not bn else INK2, linewidth=0.3, error_kw=dict(lw=0.8, ecolor=INK2),
                       label=f'{config}{" +BN" if bn else ""}')
        ax.set_xticks(range(len(sets))); ax.set_xticklabels([s.replace('_test', '').upper() for s in sets])
        ax.set_title(title, loc='left', color=INK); ax.grid(axis='x', visible=False)
    axes[0].set_ylabel('mAP')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    fig.legend(handles, labels, loc='lower center', ncol=len(ALL_CONFIGS), fontsize=8)
    savefig(fig, 'fig_real_results.png')

In [ ]:
# Per-class AP on the real datasets: baseline vs combined (+BN), and breakdowns by condition
if REAL:
    d = cls_df[(cls_df['group'] == 'all') & cls_df['eval_set'].isin([f'{r}_test' for r in REAL]) &
               cls_df['cls'].isin(CLASSES)]
    d = d.assign(variant=d['config'] + np.where(d['bn_adapt'].astype(bool), ' +BN', ''))
    pc = d[d['variant'].isin(['baseline', 'combined', 'combined +BN'])].pivot_table(
        index=['eval_set', 'cls'], columns='variant', values='AP', aggfunc='mean').round(3)
    pc.to_csv(RES / 'per_class_ap_real.csv')
    print(pc.to_string())

    g = cls_df[(cls_df['group'] != 'all') & (cls_df['cls'] == 'ALL')]
    if len(g):
        g = g.assign(variant=g['config'] + np.where(g['bn_adapt'].astype(bool), ' +BN', ''))
        by_cond = g.pivot_table(index=['eval_set', 'group'], columns='variant', values='AP', aggfunc='mean').round(3)
        by_cond.to_csv(RES / 'map_by_condition.csv')
        print('\nmAP by condition (DAWN weather / ExDark lighting type):')
        print(by_cond.to_string())

In [ ]:
# Synthetic-vs-real gap: does a gain measured on synthetic data carry over to real conditions?
PAIRS = {'rtts': ['haze'], 'exdark': ['lowlight'], 'dawn': ['haze', 'rain', 'snow', 'sandstorm']}
gap_rows = []
for config in [c for c in ALL_CONFIGS if c != 'baseline']:
    for r, kinds_r in PAIRS.items():
        if r not in REAL:
            continue
        syn_sets = [f'voc_syn_{k}_s{s}' for k in kinds_r for s in CFG['severities'] if k in CFG['degr_types']]
        def score(cfg_name, sets, bn=False):
            d = cls_df[(cls_df['cls'] == 'ALL') & (cls_df['group'] == 'all') &
                       (cls_df['bn_adapt'].astype(bool) == bn) & (cls_df['config'] == cfg_name)]
            return d[d['eval_set'].isin(sets)]['AP'].mean()
        real = [f'{r}_test']
        gap_rows.append(dict(config=config, real_dataset=r, matched_synthetic='+'.join(kinds_r),
                             gain_on_synthetic=round(score(config, syn_sets) - score('baseline', syn_sets), 4),
                             gain_on_real=round(score(config, real) - score('baseline', real), 4),
                             gain_on_real_with_BN=round(score(config, real, True) - score('baseline', real), 4)))
gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(RES / 'synthetic_vs_real_gain.csv', index=False)
print('mAP gain over baseline: synthetic proxy vs real data')
print(gap_df.to_string(index=False))

In [ ]:
# Training curves (seed 0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for k, name in enumerate(ALL_CONFIGS):
    p = ckpt_path(name, CFG['seeds'][0])
    if not p.exists():
        continue
    h = pd.DataFrame(torch.load(p, map_location='cpu')['history'])
    axes[0].plot(h['epoch'], h['loss'], '-o', color=SERIES[k], lw=2, ms=4, label=name)
    axes[1].plot(h['epoch'], h['tune_mAP'], '-o', color=SERIES[k], lw=2, ms=4, label=name)
axes[0].set_title('Training loss', loc='left', color=INK); axes[1].set_title('Clean tuning mAP', loc='left', color=INK)
for a in axes:
    a.set_xlabel('epoch')
axes[1].legend(fontsize=8)
fig.tight_layout()
savefig(fig, 'fig_training_curves.png')

## 15. Qualitative results and Grad-CAM
Real test images where the improved model (combined + BN adaptation) and the baseline disagree most on a minority class, with Grad-CAM heat maps (last stage, `layer4`) showing where each model looks.

In [ ]:
def bank_image(name, i):
    X = np.load(bank_paths(name)[0], mmap_mode='r')
    return np.array(X[i]).transpose(1, 2, 0)


def to_input(img_uint8):
    x = torch.from_numpy(np.ascontiguousarray(img_uint8.transpose(2, 0, 1))).float().div(255).unsqueeze(0)
    return ((x - MEAN) / STD).to(device)


def gradcam(model, img_uint8, cls_idx):
    feats, grads = [], []

    def fwd_hook(m, i, o):
        feats.append(o)
        o.register_hook(lambda g: grads.append(g))   # gradient of the class score w.r.t. layer4 output

    h1 = model.layer4.register_forward_hook(fwd_hook)
    model.eval(); model.zero_grad()
    with torch.enable_grad():
        out = model(to_input(img_uint8))
        out[0, cls_idx].backward()
    h1.remove()
    A, G = feats[0][0].detach(), grads[0][0].detach()
    cam = F.relu((G.mean(dim=(1, 2))[:, None, None] * A).sum(0))
    cam = cam / (cam.max() + 1e-8)
    return cv2.resize(cam.cpu().numpy(), (img_uint8.shape[1], img_uint8.shape[0]))


def overlay(img, cam):
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)[..., ::-1]
    return (0.55 * img + 0.45 * heat).astype(np.uint8)


s0 = CFG['seeds'][0]
picks = []
for r in REAL:
    pb, pi = PRED / f'baseline_s{s0}_{r}_test.npy', PRED / f'combined_s{s0}_{r}_test_bn.npy'
    if not (pb.exists() and pi.exists()):
        continue
    Pb, Pi, Y = np.load(pb).astype(np.float32), np.load(pi).astype(np.float32), bank_Y(f'{r}_test')
    for c in MIN_IDX:
        pos = np.where(Y[:, c] == 1)[0]
        if len(pos):
            j = pos[np.argmax(Pi[pos, c] - Pb[pos, c])]
            picks.append((r, int(j), c, float(Pb[j, c]), float(Pi[j, c])))
picks = sorted(picks, key=lambda t: t[4] - t[3], reverse=True)[:4]

if picks:
    m_base = load_classifier('baseline', s0)
    m_comb = load_classifier('combined', s0)
    adapted = {r: bn_adapt(m_comb, f'{r}_adapt') for r in REAL}
    fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3.1 * len(picks)))
    axes = np.atleast_2d(axes)
    for row, (r, j, c, pb_, pi_) in enumerate(picks):
        img = bank_image(f'{r}_test', j)
        axes[row, 0].imshow(img); axes[row, 0].set_title(f'{r.upper()} — true: {CLASSES[c]}', fontsize=9, color=INK)
        axes[row, 1].imshow(overlay(img, gradcam(m_base, img, c)))
        axes[row, 1].set_title(f'baseline  p({CLASSES[c]})={pb_:.2f}', fontsize=9, color=INK)
        axes[row, 2].imshow(overlay(img, gradcam(adapted[r], img, c)))
        axes[row, 2].set_title(f'combined +BN  p({CLASSES[c]})={pi_:.2f}', fontsize=9, color=INK)
    for a in axes.ravel():
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
    fig.tight_layout()
    savefig(fig, 'fig_qualitative_gradcam.png')

## 16. Localization: YOLOv8 on the same imbalanced subset
The detector is **YOLOv8n** (Ultralytics), fine-tuned from COCO weights on the same 10 classes and the same imbalanced training images. Labels are converted from VOC/ExDark boxes to YOLO format (class, centre x, centre y, width, height, normalized); VOC *difficult* boxes are dropped, as in the standard VOC protocol.

Detection configurations:
* **baseline**: the imbalanced subset as is.
* **oversample**: repeat-factor sampling (Gupta et al., LVIS, CVPR 2019): an image is repeated $r=\max_c \max(1, \sqrt{t/f_c})$ times, where $f_c$ is the fraction of training images containing class $c$ and $t$ = `rfs_t`.
* **augment**: each training image gets one extra copy with a random synthetic degradation (boxes unchanged: appearance-only shift).
* **combined**: both.

For a fair comparison, each config trains for the **same number of iterations** (epochs are scaled by dataset size). Caveat for the report: COCO pre-training has already seen these classes, which softens the effect of the imbalance.

In [ ]:
def image_size(rec):
    if not rec.get('w') or not rec.get('h'):
        with Image.open(rec['path']) as im:
            rec['w'], rec['h'] = im.size
    return rec['w'], rec['h']


def yolo_lines(rec):
    w, h = image_size(rec)
    lines = []
    for c, x1, y1, x2, y2, diff in rec['objects']:
        if c not in CIDX or diff:
            continue
        x1, x2 = sorted((min(max(x1, 0), w), min(max(x2, 0), w)))
        y1, y2 = sorted((min(max(y1, 0), h), min(max(y2, 0), h)))
        if x2 - x1 < 1 or y2 - y1 < 1:
            continue
        lines.append(f'{CIDX[c]} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}')
    return lines


def needs_reencode(path):
    """Images with an EXIF rotation are re-saved without it so pixels match the annotation frame."""
    try:
        with Image.open(path) as im:
            return im.getexif().get(0x0112, 1) not in (1, None)
    except Exception:
        return True


def link_or_copy(src, dst):
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def write_yolo_set(name, items):
    """items: list of (record, None | (kind, severity, seed)). Builds YOLO_DIR/name/{images,labels}."""
    d = YOLO_DIR / name
    if (d / '.done').exists():
        return d
    if d.exists():
        shutil.rmtree(d)
    (d / 'images').mkdir(parents=True); (d / 'labels').mkdir()

    def write_one(k):
        rec, deg = items[k]
        stem, src = f'{k:06d}', Path(rec['path'])
        if deg is None and not needs_reencode(src):
            link_or_copy(src, d / 'images' / f'{stem}{src.suffix.lower()}')
        else:
            arr = np.asarray(Image.open(src).convert('RGB'))   # raw orientation = annotation frame
            if deg is not None:
                kind, sev, sd = deg
                arr = apply_degradation(arr, kind, sev, np.random.default_rng(sd))
            Image.fromarray(arr).save(d / 'images' / f'{stem}.jpg', quality=95)
        (d / 'labels' / f'{stem}.txt').write_text('\n'.join(yolo_lines(rec)))

    with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as ex:
        list(tqdm(ex.map(write_one, range(len(items))), total=len(items), desc=f'yolo {name}', leave=False))
    (d / '.done').touch()
    return d


def det_train_items(rfs, aug, seed=0):
    rng = np.random.default_rng(seed)
    frac = class_pos_counts(train_records) / len(train_records)
    r_c = np.maximum(1.0, np.sqrt(CFG['rfs_t'] / np.maximum(frac, 1e-6)))
    items = []
    for rec in train_records:
        n = 1
        if rfs:
            r = max([r_c[CIDX[c]] for c in rec['labels'] if c in CIDX] or [1.0])
            n = int(r) + int(rng.random() < r - int(r))
        for _ in range(n):
            items.append((rec, None))
            if aug:
                kind = CFG['degr_types'][rng.integers(len(CFG['degr_types']))]
                items.append((rec, (kind, float(rng.uniform(1, 3)), int(rng.integers(1 << 31)))))
    return items


DET_SPECS = {'baseline': (False, False), 'oversample': (True, False), 'augment': (False, True), 'combined': (True, True)}
det_train_sets = {}
for cfg_name in CFG['det_configs']:
    items = det_train_items(*DET_SPECS[cfg_name])
    write_yolo_set(f'train_{cfg_name}', items)
    det_train_sets[cfg_name] = len(items)
write_yolo_set('voc_tune', [(r, None) for r in tune_records])

DET_TESTS = {'voc_test': [(r, None) for r in test_records]}
det_syn = syn_records[:CFG['det_syn_n']]
DET_TESTS['voc_syn_clean'] = [(r, None) for r in det_syn]
for kind in CFG['degr_types']:
    for sev in CFG['det_severities']:   # same per-image seeds as the classification banks
        DET_TESTS[f'voc_syn_{kind}_s{sev}'] = [
            (r, (kind, sev, [CFG['data_seed'], i, DEG_ID[kind], int(round(sev * 10))])) for i, r in enumerate(det_syn)]
for r, d in REAL.items():
    DET_TESTS[f'{r}_test'] = [(rec, None) for rec in d['test']]
    if r == 'dawn':
        for cond in sorted({rec['meta'].get('condition') for rec in d['test']}):
            sub = [(rec, None) for rec in d['test'] if rec['meta'].get('condition') == cond]
            if len(sub) >= 20:
                DET_TESTS[f'dawn_test_{cond}'] = sub
for name, items in DET_TESTS.items():
    write_yolo_set(name, items)
print('YOLO training sets (images incl. repeats/augmented copies):', det_train_sets)
print('YOLO test sets:', {k: len(v) for k, v in DET_TESTS.items()})


def yolo_yaml(train_name, test_name='voc_test'):
    p = YOLO_DIR / f'data_{train_name}__{test_name}.yaml'
    names = ''.join(f'  {i}: {c}\n' for i, c in enumerate(CLASSES))
    p.write_text(f'path: {YOLO_DIR}\ntrain: {train_name}/images\nval: voc_tune/images\n'
                 f'test: {test_name}/images\nnames:\n{names}')
    return str(p)

## 17. Train the detectors

In [ ]:
from ultralytics import YOLO


def det_ckpt(cfg_name):
    return CKPT / f'det_{cfg_name}.pt'


def train_detector(cfg_name):
    if det_ckpt(cfg_name).exists():
        return det_ckpt(cfg_name)
    n_base = det_train_sets.get('baseline', len(train_records))
    epochs = max(1, int(round(CFG['yolo_epochs'] * n_base / det_train_sets[cfg_name])))
    run = YOLO_RUNS / cfg_name
    last = run / 'weights' / 'last.pt'
    t0, save_dir = time.time(), None
    if last.exists():                      # a previous attempt exists (e.g. Colab disconnected)
        try:
            ck = torch.load(last, map_location='cpu', weights_only=False)
            resumable = ck.get('optimizer') is not None and ck.get('epoch', -1) >= 0
        except Exception:
            resumable = False
        if resumable:
            print(f'{cfg_name}: resuming interrupted training')
            model = YOLO(str(last))
            model.train(resume=True)
            save_dir = Path(model.trainer.save_dir)
        elif (run / 'weights' / 'best.pt').exists():
            print(f'{cfg_name}: previous run already finished; using its best.pt')
            save_dir = run
        else:
            shutil.rmtree(run, ignore_errors=True)
    if save_dir is None:
        model = YOLO(CFG['yolo_model'])
        model.train(data=yolo_yaml(f'train_{cfg_name}'), epochs=epochs, imgsz=CFG['yolo_imgsz'],
                    batch=CFG['yolo_batch'], seed=0, workers=CFG['num_workers'], cos_lr=True,
                    project=str(YOLO_RUNS), name=cfg_name, exist_ok=True, plots=True, verbose=False,
                    device=0 if AMP else 'cpu')
        save_dir = Path(model.trainer.save_dir)
    best = save_dir / 'weights' / 'best.pt'
    if not best.exists():
        best = save_dir / 'weights' / 'last.pt'
    shutil.copy2(best, det_ckpt(cfg_name))
    print(f'Detector {cfg_name}: {epochs} epochs, {(time.time() - t0) / 60:.1f} min, saved to {det_ckpt(cfg_name)}')
    return det_ckpt(cfg_name)


for cfg_name in CFG['det_configs']:
    train_detector(cfg_name)

## 18. Detection results
AP at IoU 0.5 (VOC protocol) per class, averaged into mAP@0.5 over the classes each test set annotates; mAP@[.5:.95] is also reported. Majority/minority means are computed from the per-class APs.

In [ ]:
DET_CSV = RES / 'det_results.csv'
det_df = pd.read_csv(DET_CSV) if DET_CSV.exists() else pd.DataFrame()
det_done = set(zip(det_df['config'], det_df['eval_set'])) if len(det_df) else set()
for cfg_name in CFG['det_configs']:
    model = YOLO(str(det_ckpt(cfg_name)))
    for test_name in DET_TESTS:
        if (cfg_name, test_name) in det_done:
            continue
        r = model.val(data=yolo_yaml(f'train_{cfg_name}', test_name), split='test', imgsz=CFG['yolo_imgsz'],
                      batch=32, conf=0.001, iou=0.6, plots=False, verbose=False,
                      project=str(DATA / 'yolo_val'), name=f'{cfg_name}_{test_name}', exist_ok=True,
                      device=0 if AMP else 'cpu')
        per = {CLASSES[int(i)]: float(ap) for i, ap in zip(r.box.ap_class_index, r.box.ap50)}
        rows = [dict(config=cfg_name, eval_set=test_name, cls=c, AP50=v) for c, v in per.items()]
        for tag, names in [('MAJ', MAJORITY), ('MIN', MINORITY)]:
            vals = [per[c] for c in names if c in per]
            if vals:
                rows.append(dict(config=cfg_name, eval_set=test_name, cls=tag, AP50=float(np.mean(vals))))
        rows.append(dict(config=cfg_name, eval_set=test_name, cls='ALL', AP50=float(r.box.map50),
                         AP50_95=float(r.box.map)))
        det_df = pd.concat([det_df, pd.DataFrame(rows)], ignore_index=True)
        det_df.to_csv(DET_CSV, index=False)

det_tables = {}
for tag, title in [('ALL', 'mAP@0.5, all classes'), ('MIN', 'mAP@0.5, minority'), ('MAJ', 'mAP@0.5, majority')]:
    d = det_df[det_df['cls'] == tag].copy()
    syn = d['eval_set'].str.match(r'voc_syn_.+_s\d')
    d.loc[syn, 'eval_set'] = 'syn avg'
    t = d.groupby(['config', 'eval_set'])['AP50'].mean().unstack().round(3)
    order = ['voc_test', 'syn avg'] + [k for k in DET_TESTS if k.endswith('_test') or k.startswith('dawn_test_')]
    order = [c for c in dict.fromkeys(order) if c in t.columns]
    det_tables[title] = t.reindex([c for c in CFG['det_configs'] if c in t.index])[order]
    print(f'\n=== Detection {title} ===')
    print(det_tables[title].to_string())
    det_tables[title].to_csv(RES / f'det_table_{tag}.csv')

In [ ]:
# Detection qualitative: one real image per dataset, baseline vs best config
show_cfgs = [c for c in ['baseline', 'combined'] if det_ckpt(c).exists()] or list(CFG['det_configs'])[:2]
samples = []
for r, d in REAL.items():
    with_min = [rec for rec in d['test'] if set(rec['labels']) & set(MINORITY)]
    if with_min:
        samples.append((r, random.Random(1).choice(with_min)))
if samples and show_cfgs:
    dets = {c: YOLO(str(det_ckpt(c))) for c in show_cfgs}
    fig, axes = plt.subplots(len(samples), len(show_cfgs), figsize=(5.2 * len(show_cfgs), 3.8 * len(samples)))
    axes = np.array(axes).reshape(len(samples), len(show_cfgs))
    for i, (r, rec) in enumerate(samples):
        img = np.asarray(load_rgb(rec['path'], 1024))
        for j, c in enumerate(show_cfgs):
            res = dets[c].predict(img[..., ::-1].copy(), imgsz=CFG['yolo_imgsz'], conf=0.25, verbose=False)[0]
            axes[i, j].imshow(res.plot()[..., ::-1])
            axes[i, j].set_title(f'{r.upper()} — {c}', fontsize=9, color=INK)
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([]); axes[i, j].grid(False)
    fig.tight_layout()
    savefig(fig, 'fig_detection_qualitative.png')

## 19. Novelty preview (off by default)
**Idea: minority-weighted clean–degraded consistency.** Each training image is seen as a clean view and a randomly degraded view with identical geometry. The loss is the mean BCE of both views plus a Jensen–Shannon consistency term between their per-class predictions, weighted by the class-balanced weights so that minority classes are pushed hardest toward degradation-invariant features. Combined with BN adaptation on unlabeled real images, the hypothesis is a larger minority-class gain on RTTS/ExDark/DAWN than weighting or augmentation alone.

To run it, set `RUN_NOVELTY_PREVIEW = True` in Section 12 and re-run Sections 12–15; it appears as the `proposed` row in every table and plot.

## 20. Interactive demo
Upload any image (or pick one of the real test examples), optionally apply a synthetic degradation, and compare the baseline with the improved models: class probabilities from the classifiers (with optional BN adaptation to a real condition) and boxes from the detectors. With `share=True` Colab prints a public link that works during the live viva.

In [ ]:
!pip install -q gradio
# Example images for the demo: a few real test images containing minority classes
examples = []
for r, d in REAL.items():
    cands = [rec for rec in d['test'] if set(rec['labels']) & set(MINORITY)][:3]
    for k, rec in enumerate(cands):
        dst = DEMO_DIR / f'{r}_{k}{Path(rec["path"]).suffix.lower()}'
        if not dst.exists():
            load_rgb(rec['path'], 1024).save(dst)
        examples.append(str(dst))
for k, rec in enumerate(test_records[:2]):
    dst = DEMO_DIR / f'voc_{k}.jpg'
    if not dst.exists():
        load_rgb(rec['path'], 1024).save(dst)
    examples.append(str(dst))


DEMO = {}   # models loaded once for the demo


def load_demo_models():
    s0 = CFG['seeds'][0]
    DEMO['cls'] = {n: load_classifier(n, s0) for n in ['baseline', 'combined'] if ckpt_path(n, s0).exists()}
    DEMO['improved'] = DEMO['cls'].get('combined') or next(iter(DEMO['cls'].values()))
    DEMO['adapted'] = {r.upper(): bn_adapt(DEMO['improved'], f'{r}_adapt') for r in REAL}
    DEMO['det'] = {c: YOLO(str(det_ckpt(c))) for c in ['baseline', 'combined'] if det_ckpt(c).exists()}


def demo_predict(img, kind='none', sev=2.0, adapt_to='none'):
    """Returns (image fed to the models, baseline probs, improved probs, baseline boxes, improved boxes)."""
    if img is None:
        return None, None, None, None, None
    arr = np.asarray(img.convert('RGB') if isinstance(img, Image.Image) else img)[..., :3].astype(np.uint8)
    if max(arr.shape[:2]) > 1024:
        s = 1024 / max(arr.shape[:2])
        arr = cv2.resize(arr, (int(arr.shape[1] * s), int(arr.shape[0] * s)), interpolation=cv2.INTER_AREA)
    if kind != 'none':
        arr = apply_degradation(arr, kind, float(sev), np.random.default_rng(0))
    x = to_input(np.asarray(Image.fromarray(arr).resize((S, S), Image.BILINEAR)))   # same resize as the banks
    with torch.no_grad():
        pb = torch.sigmoid(DEMO['cls']['baseline'](x))[0].cpu().numpy() if 'baseline' in DEMO['cls'] else None
        pi = torch.sigmoid(DEMO['adapted'].get(adapt_to, DEMO['improved'])(x))[0].cpu().numpy()

    def as_label(p):
        return None if p is None else {c: float(v) for c, v in zip(CLASSES, p)}

    def boxes(name):
        if name not in DEMO['det']:
            return None
        res = DEMO['det'][name].predict(arr[..., ::-1].copy(), imgsz=CFG['yolo_imgsz'], conf=0.25, verbose=False)[0]
        return res.plot()[..., ::-1]

    return arr, as_label(pb), as_label(pi), boxes('baseline'), boxes('combined')


def build_demo():
    import gradio as gr
    with gr.Blocks(title='IE 643 — Latent Navigators demo') as demo:
        gr.Markdown('### Classification and localization under class imbalance and extreme visual shift\n'
                    'Baseline vs improved model (class-balanced weights + degradation augmentation, '
                    'optional BN adaptation to a real condition).')
        with gr.Row():
            with gr.Column(scale=1):
                inp = gr.Image(type='numpy', label='Input image')
                kind = gr.Dropdown(['none'] + list(DEGRADATIONS), value='none', label='Synthetic degradation')
                sev = gr.Slider(1, 3, value=2, step=0.1, label='Severity')
                adapt_to = gr.Dropdown(['none'] + list(DEMO['adapted']), value='none',
                                       label='BN adaptation (improved model)')
                btn = gr.Button('Run', variant='primary')
            with gr.Column(scale=1):
                shown = gr.Image(label='Image fed to the models')
                with gr.Row():
                    lb = gr.Label(num_top_classes=5, label='Baseline classifier')
                    li = gr.Label(num_top_classes=5, label='Improved classifier')
        with gr.Row():
            db = gr.Image(label='Baseline detector')
            di = gr.Image(label='Improved detector')
        if examples:
            gr.Examples([[e, 'none', 2, 'none'] for e in examples], inputs=[inp, kind, sev, adapt_to])
        btn.click(demo_predict, [inp, kind, sev, adapt_to], [shown, lb, li, db, di])
    return demo


load_demo_models()
_check = demo_predict(Image.open(examples[0]) if examples else np.zeros((S, S, 3), np.uint8), 'haze', 2.0)
print('Demo self-check OK; top improved-model classes on the first example:',
      sorted(_check[2].items(), key=lambda kv: -kv[1])[:3])

if CFG['launch_demo']:
    demo = build_demo()
    demo.launch(share=IN_COLAB, debug=False)

## 21. Summary export for the report and slides
Writes `results/summary.md` with the dataset audit, the main classification and detection tables and the synthetic-vs-real comparison. All figures are in `figures/`.

In [ ]:
lines = ['# IE 643 — Latent Navigators: results summary', '',
         f'Hardware: {HW}', '', '## Imbalanced training subset', '',
         to_markdown(audit[['group', 'target (images)', 'realised (images)', 'realised (objects)']]), '',
         f'Training images: {len(train_records)}; majority-mean / minority-mean image ratio: {mean_ratio:.1f}', '',
         '## Real-world test sets', '', to_markdown(real_summary.set_index('dataset')) if len(real_summary) else '(none)', '']
for title, t in TABLES.items():
    lines += [f'## Classification — {title} (mean ± std over seeds {CFG["seeds"]})', '', to_markdown(t), '']
if len(gap_df):
    lines += ['## mAP gain over baseline: synthetic proxy vs real data', '', to_markdown(gap_df.set_index('config')), '']
for title, t in det_tables.items():
    lines += [f'## Detection — {title}', '', to_markdown(t), '']
(RES / 'summary.md').write_text('\n'.join(lines))
print('\n'.join(lines))
print('\nFigures:', sorted(p.name for p in FIG.glob('*.png')))